# Introduction (TBD)

# Libraries and Global Variables

In [ ]:
#Data Analysis and manupalation libraries
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from pathlib import Path

#Sklearn
from sklearn.preprocessing import LabelEncoder

#Seaborn
import seaborn as sns

#Matplotlib
import matplotlib.pyplot as plt

In [ ]:
#Seed for ML Algorithms
SEED = 81

#Paths
PROJECT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
GENERATED_REPORTS = (PROJECT / "reports" / "generated")

DIABETES_CSV = PROJECT / "data" / "raw" / "diabetic_data.csv"
IDS_MAPING_CSV = PROJECT / "data" / "raw" / "IDS_mapping.csv"


## Diabetes fact table

In [ ]:
#Read diabetes CSV
df_diabetes = pd.read_csv(DIABETES_CSV, delimiter=',')
df_diabetes.dtypes.to_csv(PROJECT / "data" / "raw" / "diabetes_dtypes.csv")

#CATEGORIZE VARIABLES

## NUMERIC
NUMERIC = [
    "encounter_id",
    "patient_nbr",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

CATEGORICAL = [
    "race",
    "gender",
    "age",
    "weight",
    "payer_code",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
    "change",
    "diabetesMed",
    "readmitted"
]

In [ ]:
df_diabetes.head()

## IDS Mapping dim tables
TODO: Add small explanation behind logic of generator 

In [ ]:
def process_ids_tables():
    #Read IDS mapping CSV
    df_ids_map = pd.read_csv(IDS_MAPING_CSV, header=None, names=range(2), delimiter=',')

    #Define table names
    table_names = ['admission_type', 'discharge_disposition_id', 'admission_source_id']

    groups = df_ids_map[0].isin(table_names).cumsum()

    tables = {g.iloc[0,0]: g.iloc[0:-1] for k,g in df_ids_map.groupby(groups)}

    for name, table in tables.items():
        #Create dataframe
        df = pd.DataFrame(table)

        #Promote first line to header
        df.columns = df.iloc[0]
        df = df.iloc[1:].reset_index(drop=True)

        yield df

ids_mapping_dataframes = process_ids_tables()

#Use generator to create our dataframes for each table

#Admission Types
df_admission_type = next(ids_mapping_dataframes)

#Cast admission_type_id to int for consistency with df_diabetes
df_admission_type['admission_type_id'] = df_admission_type['admission_type_id'].astype(int)

#Discharge Disposition
df_discharge_disposition = next(ids_mapping_dataframes)

#Cast discharge_disposition_id to int for consistency with df_diabetes
df_discharge_disposition['discharge_disposition_id'] = df_discharge_disposition['discharge_disposition_id'].astype(int)

#Admission Source
df_admission_source = next(ids_mapping_dataframes)

#Cast admission_source_id to int for consistency with df_diabetes
df_admission_source['admission_source_id'] = df_admission_source['admission_source_id'].astype(int)

### Admission Type mapping

In [ ]:
print(f"Min Id: {df_admission_type['admission_type_id'].min()}")
print(f"Max Id: {df_admission_type['admission_type_id'].max()}")

n_rows = df_admission_type.shape[0]
df_admission_type[:n_rows]

### Discharge Disposition mapping

In [ ]:
print(f"Min Id: {df_discharge_disposition['discharge_disposition_id'].min()}")
print(f"Max Id: {df_discharge_disposition['discharge_disposition_id'].max()}")

n_rows = df_discharge_disposition.shape[0]
df_discharge_disposition[:n_rows]

### Admission Source mapping

In [ ]:
print(f"Min Id: {df_admission_source['admission_source_id'].min()}")
print(f"Max Id: {df_admission_source['admission_source_id'].max()}")

n_rows = df_admission_source.shape[0]
df_admission_source[:n_rows]

# EDA

Just for starters, we'll create a Profile Report for the diabetes main fact table

In [ ]:
def create_profile_report(df, title, file_name):

    profile = ProfileReport(
        df, title = title
    )
    profile.config.html.inline = True

    path_to_html = Path(PROJECT / "reports" / "generated" / file_name)

    if not path_to_html.exists():
        profile.to_file(GENERATED_REPORTS / file_name)

#Create profile report for this dataframe
create_profile_report(df_diabetes, "Diabetes Data Profile Report", "profile-report-diabetes.html")

## Administrative Data
Here we'll explore data regarding administrative data of the patient. This includes the following columns:
- admission_type_id
- admission_source_id
- discharge_disposition_id
- payer_code
- medical_specialty


### Preview, describe, nulls

In [ ]:
#Create a list with columns of administrative data
admin_data_cols = [
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "payer_code",
    "medical_specialty"
]

#Create a daframe with only these columns
df_admin_data = df_diabetes[admin_data_cols].copy()

#check dtypes
print(df_admin_data.dtypes)

df_admin_data.tail(10)

In [ ]:
#Create profile report for this dataframe
create_profile_report(df_admin_data, "Diabetes - Administrative Data Profile Report", "profile-report-admin-data.html")        

In [ ]:
#Describe numerical columns
df_admin_data.describe()

**Note to self:** Last id 29 for discharge disposition seems like it's not being used

In [ ]:
#Describe categorical columns
df_admin_data.describe(include='object')

In [ ]:
#Percentage of nan values across dataframe
round(df_admin_data.replace('?', np.nan).isna().sum() / df_admin_data.shape[0] * 100, 2)

**Key Takeaway:** Columns payer_code and medical_specialty have high null values. 39% and 49% respectively

In [ ]:
#Check distinct values of payer code for context
pd.DataFrame(df_admin_data['payer_code'].unique())

Not much to conclude from these values. Looks like country codes but verifying some of them like "OG" they do not correspond to any country.
There is no DIM table for these column. Just keep values as is (mentioned in project guide) and later decide what to do with null values. 

### Correlation

In [ ]:
#This dataset has int and string values, so I'll encode the str values to int for an improved correlation
#Columns: payer_code, medical_specialty

le = LabelEncoder()
df_admin_data['payer_code'] = le.fit_transform(df_admin_data['payer_code'])
df_admin_data['medical_specialty'] = le.fit_transform(df_admin_data['medical_specialty'])

print(df_admin_data.dtypes)

df_admin_data[['payer_code', 'medical_specialty']].tail(10)

In [ ]:
#TODO: Qui square value of admission type and admission source

In [ ]:
#Build correlation matrix
# matrix = df_admin_data.corr()

#Buld plot
plt.figure(figsize=(6, 5))
sns.heatmap(df_admin_data.corr(), annot=True, cmap='coolwarm', )
plt.title("Correlation Heatmap")
plt.show()

## Utilization history/current stay data

In [ ]:
util_stay_cols = [
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

df_util_stay_data = df_diabetes[util_stay_cols].copy()

#dtypes
print(df_util_stay_data.dtypes)

df_util_stay_data.head(10)

In [ ]:
#Describe